# Global employment and unemployment (1991–2025)

ILOSTAT-style panel from [Kaggle: lucalullo/global-employment-unemployment-rates-1991-2025](https://www.kaggle.com/datasets/lucalullo/global-employment-unemployment-rates-1991-2025).

- **employment** (`occupazione`): `obs_value` is the employment rate for the slice.
- **unemployment** (`disoccupazione`): `obs_value` is the unemployment rate.

Dimensions: `iso_code`, `country`, `sex`, `age`, `year`. On Kaggle, add this dataset to the kernel; locally run `make download` so `data/occupazione.csv` and `data/disoccupazione.csv` exist.


## Setup & data loading

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import display, HTML

# Prism palette (same pattern as denver-cpi / wine-reviews)
PRISM = list(px.colors.qualitative.Prism)
px.defaults.color_discrete_sequence = PRISM
px.defaults.color_continuous_scale = [[i / max(len(PRISM) - 1, 1), c] for i, c in enumerate(PRISM)]
px.defaults.template = "plotly_white"

_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
if _IS_KAGGLE:
    pio.renderers.default = "iframe"


def show_plotly(fig):
    fig.update_layout(paper_bgcolor="white", plot_bgcolor="white", colorway=PRISM)
    if os.environ.get("PLOTLY_FORCE_HTML", "").lower() in ("1", "true", "yes"):
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    elif _IS_KAGGLE:
        fig.show()
    else:
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


def resolve_data_dir() -> Path:
    """Kaggle mount name matches dataset slug; fall back to search or local data/."""
    explicit = Path("/kaggle/input/global-employment-unemployment-rates-1991-2025")
    if explicit.exists():
        return explicit
    base = Path("/kaggle/input")
    if base.exists():
        for p in base.rglob("occupazione.csv"):
            return p.parent
    local = Path("data")
    assert (local / "occupazione.csv").exists() and (local / "disoccupazione.csv").exists(), (
        f"Expected occupazione.csv and disoccupazione.csv under {local.resolve()} — run `make download`"
    )
    return local


def load_employment_frames(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load employment / unemployment long tables (CSV or single Excel with Italian sheet names)."""
    xlsx = sorted(data_dir.glob("*.xlsx")) + sorted(data_dir.glob("*.xls"))
    if xlsx:
        path = xlsx[0]
        emp = pd.read_excel(path, sheet_name="occupazione")
        unemp = pd.read_excel(path, sheet_name="disoccupazione")
    else:
        ocp, disc = data_dir / "occupazione.csv", data_dir / "disoccupazione.csv"
        assert ocp.exists() and disc.exists(), f"Missing CSVs under {data_dir}"
        emp = pd.read_csv(ocp)
        unemp = pd.read_csv(disc)
    for df in (emp, unemp):
        assert set(df.columns) >= {"iso_code", "country", "sex", "age", "year", "obs_value"}
    return emp, unemp


DATA_DIR = resolve_data_dir()
print(f"DATA_DIR = {DATA_DIR}")
df_emp, df_unemp = load_employment_frames(DATA_DIR)


DATA_DIR = data


In [2]:
display(df_emp.head())
display(df_unemp.head())
df_emp.info()
df_unemp.info()


,iso_code,country,sex,age,year,obs_value
0,AFG,Afghanistan,Total,15+,2025,32.457
1,AFG,Afghanistan,Total,15-24,2025,31.419
2,AFG,Afghanistan,Total,25+,2025,33.056
3,AFG,Afghanistan,Male,15+,2025,61.038
4,AFG,Afghanistan,Male,15-24,2025,57.355


,iso_code,country,sex,age,year,obs_value
0,AFG,Afghanistan,Total,15+,2025,13.351
1,AFG,Afghanistan,Total,15-24,2025,16.785
2,AFG,Afghanistan,Total,25+,2025,11.340
3,AFG,Afghanistan,Male,15+,2025,12.503
4,AFG,Afghanistan,Male,15-24,2025,15.814


<class 'pandas.DataFrame'>
RangeIndex: 57519 entries, 0 to 57518
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   iso_code   57519 non-null  str    
 1   country    57519 non-null  str    
 2   sex        57519 non-null  str    
 3   age        57519 non-null  str    
 4   year       57519 non-null  int64  
 5   obs_value  57519 non-null  float64
dtypes: float64(1), int64(1), str(4)
memory usage: 2.6 MB
<class 'pandas.DataFrame'>
RangeIndex: 57519 entries, 0 to 57518
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   iso_code   57519 non-null  str    
 1   country    57519 non-null  str    
 2   sex        57519 non-null  str    
 3   age        57519 non-null  str    
 4   year       57519 non-null  int64  
 5   obs_value  57519 non-null  float64
dtypes: float64(1), int64(1), str(4)
memory usage: 2.6 MB


## Validation

In [3]:
KEY = ["iso_code", "sex", "age", "year"]

def check_keys(name: str, df: pd.DataFrame) -> None:
    dup = df.duplicated(KEY).sum()
    print(f"{name}: duplicate keys on {KEY} = {dup}")
    assert dup == 0, name


check_keys("employment", df_emp)
check_keys("unemployment", df_unemp)

for label, df in ("employment", df_emp), ("unemployment", df_unemp):
    print(label, "years", df["year"].min(), "–", df["year"].max(), "| countries", df["iso_code"].nunique())
    print("  sex:", sorted(df["sex"].unique()))
    print("  age:", sorted(df["age"].unique(), key=lambda x: (len(x), x)))


employment: duplicate keys on ['iso_code', 'sex', 'age', 'year'] = 0
unemployment: duplicate keys on ['iso_code', 'sex', 'age', 'year'] = 0
employment years 1991 – 2025 | countries 183
  sex: ['Female', 'Male', 'Total']
  age: ['15+', '25+', '15-24']
unemployment years 1991 – 2025 | countries 183
  sex: ['Female', 'Male', 'Total']
  age: ['15+', '25+', '15-24']


## Unweighted global trends (baseline slice)

**Mean across countries** for each year is **not** population-weighted (no population column). Baseline: `sex == "Total"` and `age == "15+"`.


In [4]:
BASE_SEX = "Total"
BASE_AGE = "15+"
assert BASE_SEX in df_unemp["sex"].unique() and BASE_AGE in df_unemp["age"].unique()

u_base = df_unemp[(df_unemp["sex"] == BASE_SEX) & (df_unemp["age"] == BASE_AGE)].copy()
e_base = df_emp[(df_emp["sex"] == BASE_SEX) & (df_emp["age"] == BASE_AGE)].copy()

unemp_year = (
    u_base.groupby("year", as_index=False)["obs_value"]
    .agg(mean_unemployment="mean", median_unemployment="median")
)
uq = u_base.groupby("year")["obs_value"].quantile([0.25, 0.75]).unstack()
uq.columns = ["q25", "q75"]
unemp_year = unemp_year.merge(uq.reset_index(), on="year")
emp_year = e_base.groupby("year", as_index=False)["obs_value"].agg(
    mean_employment="mean",
    median_employment="median",
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=unemp_year["year"], y=unemp_year["mean_unemployment"], name="Unemployment (mean)", mode="lines"))
fig.add_trace(
    go.Scatter(
        x=unemp_year["year"],
        y=unemp_year["q25"],
        name="Unemployment Q25–Q75",
        mode="lines",
        line=dict(width=0),
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=unemp_year["year"],
        y=unemp_year["q75"],
        name="Unemployment Q25–Q75",
        mode="lines",
        line=dict(width=0),
        fillcolor="rgba(99,110,250,0.2)",
        fill="tonexty",
    )
)
fig.add_trace(go.Scatter(x=emp_year["year"], y=emp_year["mean_employment"], name="Employment (mean)", mode="lines", yaxis="y2"))
fig.update_layout(
    title="Unweighted cross-country mean rates (Total, 15+) — unemployment band = Q25–Q75",
    xaxis_title="Year",
    yaxis=dict(title="Unemployment rate (%)"),
    yaxis2=dict(title="Employment rate (%)", overlaying="y", side="right"),
    hovermode="x unified",
)
show_plotly(fig)


## Stratification: sex and age

In [5]:
# Example country: Spain (adjust iso_code to explore)
ISO_FOCUS = "ESP"
u_sp = df_unemp[df_unemp["iso_code"] == ISO_FOCUS].copy()
fig = px.line(
    u_sp,
    x="year",
    y="obs_value",
    color="sex",
    facet_col="age",
    facet_col_wrap=3,
    markers=False,
    title=f"Unemployment over time by sex × age — {ISO_FOCUS} ({u_sp['country'].iloc[0]})",
)
fig.update_layout(height=400)
show_plotly(fig)


In [6]:
# Top countries by variance of unemployment (Total, 15+) in the series
uv = df_unemp[(df_unemp["sex"] == BASE_SEX) & (df_unemp["age"] == BASE_AGE)]
var_by = uv.groupby("iso_code")["obs_value"].var().sort_values(ascending=False)
TOP_N = 30
top_iso = var_by.head(TOP_N).index.tolist()
heat = uv[uv["iso_code"].isin(top_iso)].pivot_table(index="country", columns="year", values="obs_value")
heat = heat.reindex(heat.var(axis=1).sort_values(ascending=False).index)
fig = px.imshow(
    heat,
    labels=dict(x="Year", y="Country", color="Unemployment %"),
    title=f"Unemployment heatmap — top {TOP_N} countries by variance (Total, 15+)",
    aspect="auto",
    color_continuous_scale=px.colors.sequential.Blues,
)
show_plotly(fig)


## Employment vs unemployment (merged panel)

In [7]:
merged = df_emp.merge(
    df_unemp,
    on=["iso_code", "country", "sex", "age", "year"],
    suffixes=("_employment", "_unemployment"),
)
assert merged[["obs_value_employment", "obs_value_unemployment"]].notna().all().all()
print(merged.shape)
merged.head()


(57519, 7)


,iso_code,country,sex,age,year,obs_value_employment,obs_value_unemployment
0,AFG,Afghanistan,Total,15+,2025,32.457,13.351
1,AFG,Afghanistan,Total,15-24,2025,31.419,16.785
2,AFG,Afghanistan,Total,25+,2025,33.056,11.340
3,AFG,Afghanistan,Male,15+,2025,61.038,12.503
4,AFG,Afghanistan,Male,15-24,2025,57.355,15.814


In [8]:
# Scatter for baseline slice across years (color = year)
m_base = merged[(merged["sex"] == BASE_SEX) & (merged["age"] == BASE_AGE)]
fig = px.scatter(
    m_base.sample(min(8000, len(m_base)), random_state=0) if len(m_base) > 8000 else m_base,
    x="obs_value_employment",
    y="obs_value_unemployment",
    color="year",
    opacity=0.35,
    trendline="ols",
    title=f"Employment vs unemployment ({BASE_SEX}, {BASE_AGE}) — sample for readability",
    labels={"obs_value_employment": "Employment %", "obs_value_unemployment": "Unemployment %"},
)
show_plotly(fig)

r = m_base["obs_value_employment"].corr(m_base["obs_value_unemployment"])
print(f"Pearson r (employment vs unemployment, same slice, all country-years): {r:.4f}")


Pearson r (employment vs unemployment, same slice, all country-years): -0.6547


## Country narratives (Total, 15+)

In [9]:
# Countries with largest change in unemployment in the last decade (vs prior decade)
uvb = df_unemp[(df_unemp["sex"] == BASE_SEX) & (df_unemp["age"] == BASE_AGE)].copy()
last_years = sorted(uvb["year"].unique())[-10:]
prior_years = sorted(uvb["year"].unique())[-20:-10]
recent = uvb[uvb["year"].isin(last_years)].groupby("iso_code")["obs_value"].mean()
prior = uvb[uvb["year"].isin(prior_years)].groupby("iso_code")["obs_value"].mean()
delta = (recent - prior).dropna().sort_values(key=abs, ascending=False)
NARRATIVE_ISOS = delta.head(4).index.tolist()
print("Largest |Δ mean unemployment| (last 10y vs prior 10y):", NARRATIVE_ISOS)

plot_df = merged[(merged["iso_code"].isin(NARRATIVE_ISOS)) & (merged["sex"] == BASE_SEX) & (merged["age"] == BASE_AGE)]
plot_long = plot_df.melt(
    id_vars=["year", "country", "iso_code"],
    value_vars=["obs_value_employment", "obs_value_unemployment"],
    var_name="metric",
    value_name="rate",
)
plot_long["metric"] = plot_long["metric"].map(
    {"obs_value_employment": "Employment", "obs_value_unemployment": "Unemployment"}
)
fig = px.line(
    plot_long,
    x="year",
    y="rate",
    color="metric",
    facet_col="country",
    facet_col_wrap=2,
    markers=False,
    title=f"Employment & unemployment — selected countries ({BASE_SEX}, {BASE_AGE})",
)
fig.update_layout(height=450)
show_plotly(fig)


Largest |Δ mean unemployment| (last 10y vs prior 10y): ['MKD', 'BIH', 'SRB', 'ZAF']
